# Project report: Poisson Regression for MTPL Ratemaking (GLM & GAM)

## 1. Project Objective
The objective of this study was to develop a risk assessment system for Motor Third-Party Liability (MTPL) insurance in Belgium. The primary challenge was to determine whether the policyholder's place of residence significantly impacts claim frequency and to establish fair geographical tariff zones (e.g., higher premiums in large cities, lower in rural areas).

The analysis was conducted in the R environment, comparing a traditional analytical approach with one incorporating spatial geographic mapping.

## 2. Data Preparation
Raw insurance data was pre-processed to enable reliable modeling. Key steps included:

1. **Driver Age Grouping:** Quantile binning was applied. Based on this, key stages of a driver's life cycle were isolated — from the high-risk group (Young Risk, 18–25 years), through stable mature drivers, to seniors. The Adult Base segment (middle-aged drivers), characterized by the highest claim stability, was selected as the reference group.
2. **Engine Power:** The continuous variable was discretized into 10 kW intervals and then grouped into 5 logical classes: from Very_Low (< 30 kW) to Very_High (> 110 kW). Medium-power vehicles (Middle_Base) were established as the base group.
3. **Bonus-Malus Classification:** The BM scale (0-22) was capped at level 12 and grouped. The safest drivers (BM=0) form the reference group, while higher classes (e.g., BM 11-12) were merged into the BM_Top category.

## 3. Methodology Description
The ratemaking process was conducted in three stages, gradually increasing model complexity to achieve optimal data fit.

### 3.1 Stage 1: Optimization of Classical Variables
The first step involved analyzing a preliminary model based on highly granular variable splits (e.g., 20 quantile age groups, raw power intervals). This analysis revealed overparameterization — many adjacent groups did not differ statistically in risk, creating informational noise. Level aggregation techniques were applied, grouping variables into logical segments to minimize the AIC. The goal was to create a stable GLM based on the Poisson distribution to serve as a benchmark for spatial analysis.

### 3.2 Stage 2: Spatial Modeling (GAM)
A classical GLM assumes that risk is homogeneous nationwide for drivers with the same parameters. To verify this assumption, a Generalized Additive Model (GAM) was applied. This allowed for the generation of a continuous risk map (smooth surface), identifying local claim frequency extremes.

### 3.3 Stage 3: Zone Determination
The following approach was utilized:
1. GAM predictions were used as a measure of geographic risk.
2. The Fisher-Jenks algorithm was applied for map discretization — dividing the continuous surface into optimal tariff zones.
3. The statistical significance of differences between zones was verified, reducing their number to 4 final groups.
4. A final GLM was constructed by incorporating the new territorial variable (`geo_zone`).

## 4. Analysis Results

### 4.1 Does Location Matter?
Comparing the models clearly showed that incorporating location improves tariff quality, measured by the AIC (the lower, the better).

**Table 1: Model Quality Comparison**

| Model Type | Quality Metric (AIC) |
| :--- | :--- |
| Preliminary Model | 124 925.0 |
| Segmented Model (without location) | 124 911.3 |
| Spatial Model (geographic) | 124 545.3 |
| **Final Model (4 geographic zones)** | **124 510.9** |

The AIC drop of over 400 points is statistically significant. This indicates the existence of "high-risk" regions (e.g., city centers) that cannot be explained solely by the driver's profile.

### 4.2 Impact of Risk Factors
To evaluate the impact of individual features on claim risk, a unified relativity plot was generated *(reference: Figure 1 in the original document)*.

The plot consists of two elements:
* **Relativity Line (Orange):** Shows the risk multiplier. A value of 1.0 indicates the baseline (reference) level. Values above 1.0 indicate higher risk, and below — lower risk.
* **Exposure Bars (Navy Blue):** Show the group size (volume of policies with a given trait in the portfolio).

**Interpretation of Results:**
1. **Age:** The highest risk is associated with the Young_Risk group. The risk then decreases, reaching a minimum for the Pre_Senior and Senior groups, before rising slightly for the Elderly.
2. **Claim History (Bonus-Malus):** This is the strongest differentiating factor. For the BM_0 class (claim-free drivers, highest exposure), the risk is stable. For the BM_Top classes, risk rises drastically, reaching levels over twice the baseline.
3. **Engine Power:** An upward trend is observed. Low-power cars (Very_Low) are safer, while very high-power cars (Very_High) carry increased risk.
4. **Fuel:** Vehicles with diesel engines carry a higher risk than gasoline vehicles, reflected by a higher relativity line for this group.

Confidence intervals are narrower where exposure bars are high (e.g., Adult_Base, BM_0), indicating highly precise estimates for these groups.

### 4.3 Comprehensive Risk Factor Analysis
To verify model stability, a final relativity plot was created incorporating all variables, including the newly defined geographic zones *(reference: Figure 2 in the original document)*.

**Geographic Zones (New Factor):** The `geo_zone` section confirms the effectiveness of spatial modeling:
* **Zone_1 and Zone_2:** Reduced-risk zones. The relativity line is distinctly below 1.0 (approx. 0.8), meaning drivers here have about 20% lower claim frequency than the baseline.
* **Zone_3_4:** The base zone (most populous), defining standard risk.
* **Zone_5:** The highest risk zone (likely large city centers). Risk here is roughly 30% higher than the baseline (value approx. 1.3).

## 5. Tariff Zone Visualization
Based on the model, a risk map for all of Belgium was generated. The dark blue areas *(reference: Figure 3 in the original document)* indicate regions with the highest accident risk (primarily large urban agglomerations).

The continuous risk map was then discretized. Initially, using the Fisher-Jenks algorithm, 5 risk zones were defined. However, GLM analysis revealed a lack of significant statistical differences between zones 3 and 4. To optimize the model, the decision was made to reduce the number of zones to 4, ensuring full statistical significance for all geographic parameters *(reference: Figure 4 in the original document for the 4-zone price map)*.

## 6. Final Conclusions
This analysis proves that location is a crucial parameter in insurance pricing. Ignoring this factor would lead to unfair pricing — drivers from safer, rural regions would effectively subsidize drivers from large cities. 

Implementing a tariff based on the Final GLM (4 zones) is recommended as the target solution. This model proved to be the best statistically (lowest AIC).